In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("GPU is NOT available")

In [ ]:
import requests

url = "https://zenodo.org/records/3384388"

response = requests.get(url, timeout=30)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("Internet access is working.")
else:
    print("Internet request failed.")

In [ ]:
import requests

url = "https://zenodo.org/records/3384388/files/0_dB_fan.zip?download=1"

r = requests.head(url, allow_redirects=True, timeout=30)

print("Status:", r.status_code)
print("Size (GB):", int(r.headers.get("content-length", 0)) / 1e9)

In [ ]:
import zipfile
import fsspec

url = "https://zenodo.org/records/3384388/files/0_dB_fan.zip?download=1"

print("Opening remote ZIP...")

with fsspec.open(
    url,
    mode="rb",
    block_size=5 * 1024 * 1024
) as f:

    with zipfile.ZipFile(f) as z:

        print("ZIP opened successfully.")
        print("Number of files:", len(z.namelist()))

        print("\nFirst 5 files:")
        for name in z.namelist()[:5]:
            print(name)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import glob
import pandas as pd

files = glob.glob(
    "/kaggle/input/**/train.csv",
    recursive=True
)

print("Train files found:")
for f in files:
    print(f)

train_path = files[0]

base = train_path.rsplit("/", 1)[0]

train = pd.read_csv(
    f"{base}/train.csv"
)

val = pd.read_csv(
    f"{base}/validation.csv"
)

test = pd.read_csv(
    f"{base}/test.csv"
)

print("\nTrain:", len(train))
print("Validation:", len(val))
print("Test:", len(test))
print("Total:", len(train) + len(val) + len(test))

print("\nTrain labels:")
print(train["label"].value_counts())

print("\nValidation labels:")
print(val["label"].value_counts())

print("\nTest labels:")
print(test["label"].value_counts())

In [ ]:
!git clone https://github.com/microsoft/unilm.git

In [ ]:
import os

beats_dir = "/kaggle/working/unilm/beats"

print("BEATs directory exists:", os.path.exists(beats_dir))
print("\nFiles:")

for file in os.listdir(beats_dir):
    print(file)

In [ ]:
from huggingface_hub import hf_hub_download
import os

checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print("Downloading BEATs checkpoint...")

checkpoint_path = hf_hub_download(
    repo_id="nsivaku/nithin_checkpoints",
    filename="BEATs_iter3_plus_AS2M.pt",
    local_dir=checkpoint_dir
)

print("\nBEATs checkpoint downloaded successfully.")
print("Path:", checkpoint_path)
print("Size (MB):", os.path.getsize(checkpoint_path) / (1024 * 1024))

In [ ]:
import sys
import torch

beats_dir = "/kaggle/working/unilm/beats"

sys.path.insert(0, beats_dir)

from BEATs import BEATs, BEATsConfig


checkpoint_path = (
    "/kaggle/working/checkpoints/"
    "BEATs_iter3_plus_AS2M.pt"
)

print("Loading BEATs checkpoint...")

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

cfg = BEATsConfig(
    checkpoint["cfg"]
)

model = BEATs(cfg)

model.load_state_dict(
    checkpoint["model"]
)

model.eval()

print("BEATs loaded successfully.")

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()))

print("Checkpoint:", checkpoint_path)

In [ ]:
import pandas as pd
import torch
import soundfile as sf
import os

# Load one training sample
train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"
df = pd.read_csv(train_csv)

row = df.iloc[0]

print("Machine:", row["machine"])
print("Label:", row["label"])
print("Source:", row["source"])
print("File:", row["file_path"])

In [ ]:
import io
import zipfile
import fsspec
import soundfile as sf

source = row["source"]
file_path = row["file_path"]

url = f"https://zenodo.org/records/3384388/files/{source}?download=1"

print("Opening remote ZIP...")
print("Source:", source)
print("File:", file_path)

with fsspec.open(
    url,
    mode="rb",
    block_size=5 * 1024 * 1024
) as f:
    with zipfile.ZipFile(f) as z:
        with z.open(file_path) as audio_file:
            audio_bytes = audio_file.read()

audio, sample_rate = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

print("\nAudio loaded successfully!")
print("Sample rate:", sample_rate)
print("Shape:", audio.shape)
print("Duration:", len(audio) / sample_rate, "seconds")

In [ ]:
import torch
import numpy as np

# Convert 8-channel audio to mono
mono_audio = audio.mean(axis=1)

# Convert to PyTorch tensor
audio_tensor = torch.tensor(
    mono_audio,
    dtype=torch.float32
)

# Add batch dimension
audio_tensor = audio_tensor.unsqueeze(0)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
audio_tensor = audio_tensor.to(device)

print("Device:", device)
print("Input shape:", audio_tensor.shape)

# BEATs feature extraction
with torch.no_grad():
    representation = model.extract_features(
        audio_tensor,
        padding_mask=None
    )[0]

print("BEATs feature extraction successful!")
print("Feature shape:", representation.shape)

In [ ]:
import os
import io
import zipfile
import fsspec
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import soundfile as sf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================
# CONFIG
# ============================================================

BASE = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits"

TRAIN_CSV = os.path.join(BASE, "train.csv")
VAL_CSV   = os.path.join(BASE, "validation.csv")
TEST_CSV  = os.path.join(BASE, "test.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

# ============================================================
# REMOTE MIMII DATASET
# ============================================================

class RemoteMIMIIDataset(Dataset):

    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

        self.label_map = {
            "normal": 0,
            "abnormal": 1
        }

        self.zip_handles = {}

    def __len__(self):
        return len(self.df)

    def get_zip(self, source):

        if source not in self.zip_handles:

            url = (
                f"https://zenodo.org/records/3384388/files/"
                f"{source}?download=1"
            )

            remote_file = fsspec.open(
                url,
                mode="rb",
                block_size=5 * 1024 * 1024
            ).open()

            self.zip_handles[source] = zipfile.ZipFile(remote_file)

        return self.zip_handles[source]

    def __getitem__(self, index):

        row = self.df.iloc[index]

        source = row["source"]
        file_path = row["file_path"]

        # Read audio
        if source == "0_dB_pump":

            audio, sr = sf.read(
                file_path,
                dtype="float32"
            )

        else:

            z = self.get_zip(source)

            with z.open(file_path) as audio_file:

                audio_bytes = audio_file.read()

            audio, sr = sf.read(
                io.BytesIO(audio_bytes),
                dtype="float32"
            )

        # Convert multi-channel → mono
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        # Convert to tensor
        waveform = torch.tensor(
            audio,
            dtype=torch.float32
        )

        # Normalize
        max_value = waveform.abs().max()

        if max_value > 0:
            waveform = waveform / max_value

        # Make exactly 10 seconds
        TARGET_LENGTH = 160000

        if waveform.shape[0] < TARGET_LENGTH:

            waveform = torch.nn.functional.pad(
                waveform,
                (0, TARGET_LENGTH - waveform.shape[0])
            )

        else:

            waveform = waveform[:TARGET_LENGTH]

        label = self.label_map[row["label"]]

        return waveform, torch.tensor(
            label,
            dtype=torch.long
        )


# ============================================================
# LOAD DATASETS
# ============================================================

train_dataset = RemoteMIMIIDataset(TRAIN_CSV)
val_dataset   = RemoteMIMIIDataset(VAL_CSV)
test_dataset  = RemoteMIMIIDataset(TEST_CSV)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("DataLoaders created successfully.")

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")

print("Total storage:", round(total / 1024**3, 2), "GB")
print("Used storage:", round(used / 1024**3, 2), "GB")
print("Free storage:", round(free / 1024**3, 2), "GB")

In [ ]:
import os
import io
import zipfile
import fsspec
import pandas as pd
import soundfile as sf

# Load training split
train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"
df = pd.read_csv(train_csv)

# Test only 20 files
sample_df = df.head(20)

cache_dir = "/kaggle/working/audio_cache"
os.makedirs(cache_dir, exist_ok=True)

zip_files = {}

print("Caching 20 audio files...")

for i, row in sample_df.iterrows():

    source = row["source"]
    file_path = row["file_path"]

    # ------------------------------------------------
    # FIX WINDOWS PUMP PATH
    # ------------------------------------------------
    if source == "0_dB_pump":

        # Convert:
        # C:\Fault_Recognition\data\raw\0_dB_pump\pump\id_06\...
        #
        # into:
        # pump/id_06/...
        
        marker = "0_dB_pump\\"

        if marker in file_path:
            file_path = file_path.split(marker, 1)[1]

        # Convert Windows separators to ZIP separators
        file_path = file_path.replace("\\", "/")

    # ------------------------------------------------
    # OUTPUT PATH
    # ------------------------------------------------

    output_path = os.path.join(
        cache_dir,
        f"{i:05d}.wav"
    )

    if os.path.exists(output_path):
        continue

    # ------------------------------------------------
    # OPEN REMOTE ZIP
    # ------------------------------------------------

    zip_name = source if source.endswith(".zip") else source + ".zip"

    if source not in zip_files:

        url = (
            f"https://zenodo.org/records/3384388/files/"
            f"{zip_name}?download=1"
        )

        print("Opening:", zip_name)

        remote_file = fsspec.open(
            url,
            mode="rb",
            block_size=5 * 1024 * 1024
        ).open()

        zip_files[source] = zipfile.ZipFile(remote_file)

    z = zip_files[source]

    # ------------------------------------------------
    # READ WAV
    # ------------------------------------------------

    with z.open(file_path) as audio_file:
        audio_bytes = audio_file.read()

    audio, sr = sf.read(
        io.BytesIO(audio_bytes),
        dtype="float32"
    )

    # ------------------------------------------------
    # SAVE LOCALLY
    # ------------------------------------------------

    sf.write(
        output_path,
        audio,
        sr
    )

    print(
        f"{i+1}/20 cached:",
        file_path
    )

print("\nCaching complete.")

files = os.listdir(cache_dir)

total_size = sum(
    os.path.getsize(
        os.path.join(cache_dir, f)
    )
    for f in files
)

print("Cached files:", len(files))
print(
    "Cache size:",
    round(total_size / 1024**2, 2),
    "MB"
)

In [ ]:
import os

cache_dir = "/kaggle/working/audio_cache"

print("Cache directory exists:", os.path.exists(cache_dir))

if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    print("Number of cached files:", len(files))
    print("Files:", files[:10])

In [ ]:
import os
import io
import zipfile
import fsspec
import pandas as pd
import soundfile as sf

# Load training split
train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"
df = pd.read_csv(train_csv)

# Test only 20 files
sample_df = df.head(20)

cache_dir = "/kaggle/working/audio_cache"
os.makedirs(cache_dir, exist_ok=True)

zip_files = {}

print("Caching 20 audio files...")

for i, row in sample_df.iterrows():

    source = row["source"]
    file_path = row["file_path"]

    if source == "0_dB_pump":

        marker = "0_dB_pump\\"

        if marker in file_path:
            file_path = file_path.split(marker, 1)[1]

        file_path = file_path.replace("\\", "/")

    output_path = os.path.join(
        cache_dir,
        f"{i:05d}.wav"
    )

    if os.path.exists(output_path):
        continue

    zip_name = source if source.endswith(".zip") else source + ".zip"

    if source not in zip_files:

        url = (
            f"https://zenodo.org/records/3384388/files/"
            f"{zip_name}?download=1"
        )

        print("Opening:", zip_name)

        remote_file = fsspec.open(
            url,
            mode="rb",
            block_size=5 * 1024 * 1024
        ).open()

        zip_files[source] = zipfile.ZipFile(remote_file)

    z = zip_files[source]

    with z.open(file_path) as audio_file:
        audio_bytes = audio_file.read()

    audio, sr = sf.read(
        io.BytesIO(audio_bytes),
        dtype="float32"
    )

    sf.write(
        output_path,
        audio,
        sr
    )

    print(
        f"{i+1}/20 cached:",
        file_path
    )

print("\nCaching complete.")

files = os.listdir(cache_dir)

total_size = sum(
    os.path.getsize(
        os.path.join(cache_dir, f)
    )
    for f in files
)

print("Cached files:", len(files))
print(
    "Cache size:",
    round(total_size / 1024**2, 2),
    "MB"
)

In [ ]:
import os
import soundfile as sf

audio_path = "/kaggle/working/audio_cache/00000.wav"

audio, sr = sf.read(audio_path)

print("Audio loaded successfully!")
print("Sample rate:", sr)
print("Shape:", audio.shape)
print("Duration:", round(len(audio) / sr, 2), "seconds")

In [ ]:
# Check GPU
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled")

In [ ]:
!git clone -q https://github.com/microsoft/unilm.git /kaggle/working/unilm

In [ ]:
import sys

sys.path.insert(0, "/kaggle/working/unilm/beats")

from BEATs import BEATs, BEATsConfig

print("BEATs imported successfully!")

In [ ]:
import os

checkpoint_dir = "/kaggle/working/beats_checkpoint"
os.makedirs(checkpoint_dir, exist_ok=True)

print("Checkpoint directory ready:")
print(checkpoint_dir)

In [ ]:
import requests

url = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/README.md"

response = requests.get(url)

print("Status:", response.status_code)
print("README downloaded:", len(response.text), "characters")

# Show checkpoint-related lines
for line in response.text.splitlines():
    if "BEATs_iter" in line or "checkpoint" in line.lower():
        print(line)

In [ ]:
import requests
import os

checkpoint_path = "/kaggle/working/beats_checkpoint/BEATs_iter3_plus_AS2M.pt"

url = "https://1drv.ms/u/s!AqeByhGUtINrgcpke6_lRSZEKD5j2Q?e=A3FpOf&download=1"

print("Downloading BEATs Iter3+ AS2M...")

response = requests.get(url, stream=True, allow_redirects=True)

print("Status:", response.status_code)
print("Final URL:", response.url[:100])

with open(checkpoint_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

size_mb = os.path.getsize(checkpoint_path) / (1024 ** 2)

print("\nDownload complete!")
print("Checkpoint size:", round(size_mb, 2), "MB")
print("Saved to:", checkpoint_path)

In [ ]:
import torch
from BEATs import BEATs, BEATsConfig

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Checkpoint
checkpoint_path = "/kaggle/working/beats_checkpoint/BEATs_iter3_plus_AS2M.pt"

print("Loading BEATs checkpoint...")

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

# Create configuration
cfg = BEATsConfig(checkpoint["cfg"])

# Create model
beats = BEATs(cfg)

# Load pretrained weights
beats.load_state_dict(checkpoint["model"])

# Move to GPU
beats = beats.to(device)

# Evaluation mode
beats.eval()

print("BEATs loaded successfully!")
print("Device:", device)
print("Model ready.")

In [ ]:
import torch
import soundfile as sf

audio_path = "/kaggle/working/audio_cache/00000.wav"

# Load audio
audio, sr = sf.read(audio_path)

# Convert 8-channel audio to mono
if audio.ndim > 1:
    audio = audio.mean(axis=1)

# Convert to tensor
waveform = torch.tensor(
    audio,
    dtype=torch.float32,
    device=device
).unsqueeze(0)

print("Input shape:", waveform.shape)
print("Sample rate:", sr)

# BEATs inference
with torch.no_grad():
    features, padding_mask = beats.extract_features(
        waveform,
        padding_mask=None
    )

print("Features type:", type(features))
print("Features shape:", features.shape)
print("Padding mask:", padding_mask)

In [ ]:
import os
import torch
import soundfile as sf
import numpy as np

cache_dir = "/kaggle/working/audio_cache"

embeddings = []
file_names = []

print("Extracting BEATs embeddings...")
print()

for i in range(20):

    audio_path = os.path.join(
        cache_dir,
        f"{i:05d}.wav"
    )

    # Load audio
    audio, sr = sf.read(audio_path)

    # Convert multichannel → mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Convert to tensor
    waveform = torch.tensor(
        audio,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    # BEATs
    with torch.no_grad():

        features, padding_mask = beats.extract_features(
            waveform,
            padding_mask=None
        )

    # Mean pooling over time
    embedding = features.mean(dim=1)

    # Move to CPU
    embedding = embedding.squeeze(0).cpu().numpy()

    embeddings.append(embedding)
    file_names.append(f"{i:05d}.wav")

    print(
        f"{i+1}/20 | "
        f"{file_names[-1]} | "
        f"Embedding shape: {embedding.shape}"
    )

# Convert to NumPy
embeddings = np.array(embeddings)

print("\nExtraction complete!")
print("Final embedding matrix:", embeddings.shape)

In [ ]:
import os
import numpy as np

embedding_dir = "/kaggle/working/beats_embeddings"
os.makedirs(embedding_dir, exist_ok=True)

np.save(
    os.path.join(embedding_dir, "test_embeddings.npy"),
    embeddings
)

np.save(
    os.path.join(embedding_dir, "test_filenames.npy"),
    np.array(file_names)
)

print("Embeddings saved!")
print("Shape:", embeddings.shape)
print("Location:", embedding_dir)

In [ ]:
import pandas as pd

train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"

train_df = pd.read_csv(train_csv)

print("Training samples:", len(train_df))
print("\nColumns:")
print(train_df.columns.tolist())

print("\nClass distribution:")
print(train_df["label"].value_counts())

print("\nSources:")
print(train_df["source"].value_counts())

print("\nFirst 5 rows:")
display(train_df.head())

In [ ]:
import os
import io
import zipfile
import fsspec
import torch
import soundfile as sf
import numpy as np
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

BATCH_SIZE = 100

train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"

output_dir = "/kaggle/working/beats_train_embeddings"
os.makedirs(output_dir, exist_ok=True)

# Load training split
train_df = pd.read_csv(train_csv)

print("Total training files:", len(train_df))
print("Batch size:", BATCH_SIZE)
print("Number of batches:", int(np.ceil(len(train_df) / BATCH_SIZE)))

# ============================================================
# ZIP FILE CACHE
# ============================================================

zip_files = {}

def get_zip(source):

    if source in zip_files:
        return zip_files[source]

    zip_name = source if source.endswith(".zip") else source + ".zip"

    url = (
        f"https://zenodo.org/records/3384388/files/"
        f"{zip_name}?download=1"
    )

    print("\nOpening:", zip_name)

    remote_file = fsspec.open(
        url,
        mode="rb",
        block_size=5 * 1024 * 1024
    ).open()

    z = zipfile.ZipFile(remote_file)

    zip_files[source] = z

    return z


# ============================================================
# CHECK EXISTING BATCHES
# ============================================================

existing_batches = sorted([
    f for f in os.listdir(output_dir)
    if f.endswith(".npz")
])

print("\nAlready completed batches:", len(existing_batches))

# ============================================================
# PROCESS BATCHES
# ============================================================

total_batches = int(np.ceil(len(train_df) / BATCH_SIZE))

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    # Skip already completed batch
    if os.path.exists(batch_file):
        print(
            f"Batch {batch_idx + 1}/{total_batches} "
            f"already exists - skipping."
        )
        continue

    start = batch_idx * BATCH_SIZE
    end = min(
        start + BATCH_SIZE,
        len(train_df)
    )

    batch_df = train_df.iloc[start:end]

    batch_embeddings = []
    batch_indices = []

    print(
        f"\n{'='*60}\n"
        f"Processing batch {batch_idx + 1}/{total_batches}"
        f" | files {start} to {end-1}"
        f"\n{'='*60}"
    )

    for idx, row in batch_df.iterrows():

        source = row["source"]
        file_path = row["file_path"]

        # Fix possible Windows pump paths
        if source == "0_dB_pump":

            marker = "0_dB_pump\\"

            if marker in file_path:
                file_path = file_path.split(
                    marker,
                    1
                )[1]

            file_path = file_path.replace(
                "\\",
                "/"
            )

        # Get ZIP
        z = get_zip(source)

        # Read WAV directly from ZIP
        with z.open(file_path) as audio_file:

            audio_bytes = audio_file.read()

        audio, sr = sf.read(
            io.BytesIO(audio_bytes),
            dtype="float32"
        )

        # Multichannel → mono
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        # Tensor
        waveform = torch.tensor(
            audio,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        # BEATs
        with torch.no_grad():

            features, padding_mask = (
                beats.extract_features(
                    waveform,
                    padding_mask=None
                )
            )

        # Mean pooling
        embedding = features.mean(
            dim=1
        )

        embedding = (
            embedding
            .squeeze(0)
            .cpu()
            .numpy()
        )

        batch_embeddings.append(
            embedding
        )

        batch_indices.append(
            idx
        )

        if len(batch_embeddings) % 10 == 0:

            print(
                f"  {len(batch_embeddings)}/"
                f"{len(batch_df)} processed"
            )

    # Convert to NumPy
    batch_embeddings = np.stack(
        batch_embeddings
    )

    batch_indices = np.array(
        batch_indices
    )

    # Save checkpoint
    np.savez_compressed(
        batch_file,
        embeddings=batch_embeddings,
        indices=batch_indices
    )

    print(
        f"\nBatch {batch_idx + 1} saved!"
    )

    print(
        "Embedding shape:",
        batch_embeddings.shape
    )

print("\nALL BATCHES COMPLETE!")

In [ ]:
import os
import glob

embedding_dir = "/kaggle/working/beats_train_embeddings"

print("Directory exists:", os.path.exists(embedding_dir))

files = sorted(glob.glob(embedding_dir + "/*"))

print("Number of files:", len(files))
print("\nFirst 10 files:")
for f in files[:10]:
    print(os.path.basename(f))

print("\nLast 5 files:")
for f in files[-5:]:
    print(os.path.basename(f))

In [ ]:
import os

embedding_dir = "/kaggle/working/beats_train_embeddings"
os.makedirs(embedding_dir, exist_ok=True)

print("Directory ready:", embedding_dir)

In [ ]:
import os
import shutil

# Remove incomplete BEATs repository if it exists
repo_path = "/kaggle/working/unilm"

if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

print("Old repository removed.")

In [ ]:
!git clone https://github.com/microsoft/unilm.git /kaggle/working/unilm

In [ ]:
import os

beats_file = "/kaggle/working/unilm/beats/BEATs.py"

print("BEATs.py exists:", os.path.exists(beats_file))

In [ ]:
import sys
import torch

# Import BEATs
sys.path.insert(0, "/kaggle/working/unilm/beats")

from BEATs import BEATs, BEATsConfig

# Checkpoint path
checkpoint_path = "/kaggle/working/beats_checkpoint/BEATs_iter3_plus_AS2M.pt"

# Check checkpoint exists
import os
print("Checkpoint exists:", os.path.exists(checkpoint_path))

# Load checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

# Create model
cfg = BEATsConfig(checkpoint["cfg"])
beats = BEATs(cfg)

# Load pretrained weights
beats.load_state_dict(checkpoint["model"])

# GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

beats = beats.to(device)
beats.eval()

print("BEATs loaded successfully!")
print("Device:", device)

In [ ]:
import os
import sys
import importlib.util

beats_dir = "/kaggle/working/unilm/beats"
beats_file = os.path.join(beats_dir, "BEATs.py")

print("BEATs file exists:", os.path.exists(beats_file))
print("BEATs file:", beats_file)

# Load BEATs.py directly
spec = importlib.util.spec_from_file_location("BEATs", beats_file)
BEATs_module = importlib.util.module_from_spec(spec)

sys.modules["BEATs"] = BEATs_module
spec.loader.exec_module(BEATs_module)

BEATs = BEATs_module.BEATs
BEATsConfig = BEATs_module.BEATsConfig

print("BEATs imported successfully!")

In [ ]:
import sys
import os

beats_dir = "/kaggle/working/unilm/beats"

print("Folder exists:", os.path.exists(beats_dir))
print("BEATs.py exists:", os.path.exists(os.path.join(beats_dir, "BEATs.py")))
print("backbone.py exists:", os.path.exists(os.path.join(beats_dir, "backbone.py")))

# Put the folder at the VERY FRONT of Python's search path
sys.path.insert(0, beats_dir)

print("\nPython path:")
print(sys.path[:3])

# Check what Python can see
print("\nFiles in BEATs folder:")
print(os.listdir(beats_dir)[:20])

# Import
from BEATs import BEATs, BEATsConfig

print("\n✅ BEATs imported successfully!")

In [ ]:
import os
import sys
import importlib

beats_dir = "/kaggle/working/unilm/beats"

# Remove previous failed imports
for module_name in ["BEATs", "backbone", "modules", "quantizer", "Tokenizer"]:
    sys.modules.pop(module_name, None)

# Change into the BEATs directory temporarily
old_dir = os.getcwd()
os.chdir(beats_dir)

try:
    # Make current directory available
    if beats_dir not in sys.path:
        sys.path.insert(0, beats_dir)

    import BEATs as beats_module

    BEATs = beats_module.BEATs
    BEATsConfig = beats_module.BEATsConfig

    print("✅ BEATs imported successfully!")

finally:
    os.chdir(old_dir)

In [ ]:
import os

print("Kaggle working directory:")
print(os.listdir("/kaggle/working")[:20])

print("\nInput datasets:")
print(os.listdir("/kaggle/input"))

In [ ]:
import os

embedding_dir = "/kaggle/working/beats_train_embeddings"

print("Embedding directory exists:", os.path.exists(embedding_dir))

if os.path.exists(embedding_dir):
    files = sorted(os.listdir(embedding_dir))

    print("Number of saved files:", len(files))

    print("\nFirst 10 files:")
    for f in files[:10]:
        print(f)

    print("\nLast 10 files:")
    for f in files[-10:]:
        print(f)

In [ ]:
import os

checkpoint_dir = "/kaggle/working/checkpoints"

print("Checkpoint folder exists:", os.path.exists(checkpoint_dir))

if os.path.exists(checkpoint_dir):
    for f in os.listdir(checkpoint_dir):
        path = os.path.join(checkpoint_dir, f)
        print(
            f,
            "→",
            round(os.path.getsize(path) / (1024**2), 2),
            "MB"
        )
    

In [ ]:
import os
import sys

beats_dir = "/kaggle/working/unilm/beats"

# Make BEATs directory the current working directory
os.chdir(beats_dir)

# Also put it first in Python's search path
if beats_dir not in sys.path:
    sys.path.insert(0, beats_dir)

# Remove previous failed imports
for module_name in ["BEATs", "backbone"]:
    sys.modules.pop(module_name, None)

print("Current directory:", os.getcwd())
print("BEATs.py exists:", os.path.exists("BEATs.py"))
print("backbone.py exists:", os.path.exists("backbone.py"))

# Now import
from BEATs import BEATs, BEATsConfig

print("\n✅ BEATs imported successfully!")

In [ ]:
import sys
import os
import importlib

beats_dir = "/kaggle/working/unilm/beats"

# Force BEATs directory to the VERY FRONT of Python's path
sys.path.insert(0, beats_dir)

# Clear any previous failed/cached imports
for name in ["BEATs", "backbone"]:
    if name in sys.modules:
        del sys.modules[name]

importlib.invalidate_caches()

print("BEATs directory:", beats_dir)
print("BEATs.py:", os.path.exists(os.path.join(beats_dir, "BEATs.py")))
print("backbone.py:", os.path.exists(os.path.join(beats_dir, "backbone.py")))
print("\nFirst Python path:", sys.path[0])

# Import backbone FIRST
import backbone

print("✅ backbone imported")

# Then import BEATs
from BEATs import BEATs, BEATsConfig

print("✅ BEATs imported successfully!")

In [ ]:
import torch
import os

checkpoint_path = "/kaggle/working/checkpoints/BEATs_iter3_plus_AS2M.pt"

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print(
    "Checkpoint size:",
    round(os.path.getsize(checkpoint_path) / (1024**2), 2),
    "MB"
)

# Load checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

# Build BEATs model
cfg = BEATsConfig(checkpoint["cfg"])
beats = BEATs(cfg)

# Load pretrained weights
beats.load_state_dict(checkpoint["model"])

# GPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

beats = beats.to(device)
beats.eval()

print("\n✅ BEATs loaded successfully!")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
import zipfile
import io
import soundfile as sf
import fsspec
import torch

# Load training split
train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"
train_df = pd.read_csv(train_csv)

row = train_df.iloc[0]

print("Machine:", row["machine"])
print("Label:", row["label"])
print("Source:", row["source"])
print("File:", row["file_path"])

# Remote ZIP
zip_name = row["source"]
zip_url = f"https://zenodo.org/records/3384388/files/{zip_name}?download=1"

print("\nOpening ZIP...")

with fsspec.open(zip_url, "rb") as f:
    zip_bytes = f.read()

with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as z:
    audio_bytes = z.read(row["file_path"])

# Read WAV
audio, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

print("Sample rate:", sr)
print("Original shape:", audio.shape)

# Convert multi-channel → mono
if audio.ndim > 1:
    audio = audio.mean(axis=1)

print("Mono shape:", audio.shape)

# Tensor
waveform = torch.from_numpy(audio).float()
waveform = waveform.unsqueeze(0).to(device)

# BEATs inference
with torch.no_grad():
    features, padding_mask = beats.extract_features(
        waveform,
        padding_mask=None
    )

print("\nBEATs feature shape:", features.shape)

# Mean pooling
embedding = features.mean(dim=1)

print("Embedding shape:", embedding.shape)
print("\n✅ SINGLE AUDIO TEST SUCCESSFUL!")

In [ ]:
import os

base = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)

    if level <= 2:
        print(root, "→", len(files), "files")

        for f in files[:10]:
            print("   ", f)

In [ ]:
import os

base = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits"

total_size = 0
file_count = 0

for root, dirs, files in os.walk(base):
    for f in files:
        path = os.path.join(root, f)
        total_size += os.path.getsize(path)
        file_count += 1

print("Files:", file_count)
print("Total size:", round(total_size / (1024**3), 2), "GB")

In [ ]:
import os
import io
import zipfile
import fsspec
import torch
import soundfile as sf
import numpy as np
import pandas as pd
import json

# ============================================================
# SETTINGS
# ============================================================

BATCH_SIZE = 100

train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"

output_dir = "/kaggle/working/beats_train_embeddings"
os.makedirs(output_dir, exist_ok=True)

progress_file = os.path.join(
    output_dir,
    "progress.json"
)

# Load training split
train_df = pd.read_csv(train_csv)

total_batches = int(
    np.ceil(len(train_df) / BATCH_SIZE)
)

print("Total training files:", len(train_df))
print("Batch size:", BATCH_SIZE)
print("Number of batches:", total_batches)


# ============================================================
# REMOTE ZIP CACHE
# ============================================================

zip_files = {}

def get_zip(source):

    if source in zip_files:
        return zip_files[source]

    zip_name = (
        source
        if source.endswith(".zip")
        else source + ".zip"
    )

    url = (
        f"https://zenodo.org/records/3384388/files/"
        f"{zip_name}?download=1"
    )

    print("\nOpening:", zip_name)

    remote_file = fsspec.open(
        url,
        mode="rb",
        block_size=5 * 1024 * 1024
    ).open()

    z = zipfile.ZipFile(remote_file)

    zip_files[source] = z

    return z


# ============================================================
# CHECK COMPLETED BATCHES
# ============================================================

completed = []

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    if os.path.exists(batch_file):
        completed.append(batch_idx)

print(
    "\nAlready completed:",
    len(completed),
    "/",
    total_batches
)

if completed:
    print(
        "Completed batches:",
        [x + 1 for x in completed]
    )


# ============================================================
# PROCESS
# ============================================================

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    # --------------------------------------------------------
    # Skip completed batches
    # --------------------------------------------------------

    if os.path.exists(batch_file):

        print(
            f"\nBatch {batch_idx + 1}/{total_batches} "
            f"already saved → SKIPPING"
        )

        continue


    start = batch_idx * BATCH_SIZE

    end = min(
        start + BATCH_SIZE,
        len(train_df)
    )

    batch_df = train_df.iloc[start:end]


    print("\n" + "=" * 60)

    print(
        f"Processing batch {batch_idx + 1}/{total_batches}"
    )

    print(
        f"Files {start} to {end - 1}"
    )

    print("=" * 60)


    batch_embeddings = []
    batch_indices = []


    # --------------------------------------------------------
    # PROCESS EACH AUDIO
    # --------------------------------------------------------

    for idx, row in batch_df.iterrows():

        source = row["source"]

        file_path = row["file_path"]


        # Pump path correction
        if source == "0_dB_pump":

            marker = "0_dB_pump\\"

            if marker in file_path:

                file_path = file_path.split(
                    marker,
                    1
                )[1]

            file_path = file_path.replace(
                "\\",
                "/"
            )


        # Get remote ZIP
        z = get_zip(source)


        # Read ONLY required WAV
        with z.open(file_path) as audio_file:

            audio_bytes = audio_file.read()


        # WAV → NumPy
        audio, sr = sf.read(
            io.BytesIO(audio_bytes),
            dtype="float32"
        )


        # 8-channel → mono
        if audio.ndim > 1:

            audio = audio.mean(
                axis=1
            )


        # NumPy → GPU tensor
        waveform = torch.tensor(
            audio,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)


        # BEATs
        with torch.no_grad():

            features, padding_mask = (
                beats.extract_features(
                    waveform,
                    padding_mask=None
                )
            )


        # Mean pooling
        embedding = features.mean(
            dim=1
        )


        # GPU → CPU
        embedding = (
            embedding
            .squeeze(0)
            .cpu()
            .numpy()
        )


        batch_embeddings.append(
            embedding
        )

        batch_indices.append(
            idx
        )


        if len(batch_embeddings) % 10 == 0:

            print(
                f"  {len(batch_embeddings)}/"
                f"{len(batch_df)} processed"
            )


    # --------------------------------------------------------
    # SAVE BATCH
    # --------------------------------------------------------

    batch_embeddings = np.stack(
        batch_embeddings
    )

    batch_indices = np.array(
        batch_indices
    )


    np.savez_compressed(
        batch_file,
        embeddings=batch_embeddings,
        indices=batch_indices
    )


    # --------------------------------------------------------
    # UPDATE PROGRESS
    # --------------------------------------------------------

    completed_now = []

    for b in range(total_batches):

        f = os.path.join(
            output_dir,
            f"batch_{b:04d}.npz"
        )

        if os.path.exists(f):
            completed_now.append(b)


    with open(
        progress_file,
        "w"
    ) as f:

        json.dump(
            {
                "total_batches": total_batches,
                "completed_batches": completed_now,
                "total_files": len(train_df)
            },
            f,
            indent=2
        )


    print(
        f"\n✅ Batch {batch_idx + 1} SAVED!"
    )

    print(
        "Embedding shape:",
        batch_embeddings.shape
    )

    print(
        "Progress:",
        len(completed_now),
        "/",
        total_batches
    )


print("\n" + "=" * 60)
print("🎉 ALL BATCHES COMPLETE!")
print("=" * 60)

In [ ]:
import os

output_dir = "/kaggle/working/beats_train_embeddings"

files = sorted([
    f for f in os.listdir(output_dir)
    if f.endswith(".npz")
])

print("Saved batch files:", len(files))
print(files)

In [ ]:
import shutil
import os

output_dir = "/kaggle/working/beats_train_embeddings"

checkpoint = "/kaggle/working/BEATs_checkpoint_36_batches"

shutil.make_archive(
    checkpoint,
    'zip',
    output_dir
)

zip_path = checkpoint + ".zip"

print("✅ CHECKPOINT CREATED")
print("File:", zip_path)
print(
    "Size:",
    round(os.path.getsize(zip_path) / (1024**2), 2),
    "MB"
)

In [ ]:
import os
import io
import zipfile
import fsspec
import torch
import soundfile as sf
import numpy as np
import pandas as pd
import json

BATCH_SIZE = 100

train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"
output_dir = "/kaggle/working/beats_train_embeddings"

os.makedirs(output_dir, exist_ok=True)

progress_file = os.path.join(output_dir, "progress.json")

train_df = pd.read_csv(train_csv)

total_batches = int(np.ceil(len(train_df) / BATCH_SIZE))

print("Total training files:", len(train_df))
print("Batch size:", BATCH_SIZE)
print("Number of batches:", total_batches)


# --------------------------------------------------
# OPEN REMOTE ZIP FILES
# --------------------------------------------------

zip_files = {}

def get_zip(source):

    if source in zip_files:
        return zip_files[source]

    zip_name = source if source.endswith(".zip") else source + ".zip"

    url = (
        f"https://zenodo.org/records/3384388/files/"
        f"{zip_name}?download=1"
    )

    print("\nOpening:", zip_name)

    remote_file = fsspec.open(
        url,
        mode="rb",
        block_size=5 * 1024 * 1024
    ).open()

    z = zipfile.ZipFile(remote_file)

    zip_files[source] = z

    return z


# --------------------------------------------------
# FIND ALREADY COMPLETED BATCHES
# --------------------------------------------------

completed = []

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    if os.path.exists(batch_file):
        completed.append(batch_idx)


print(
    "\nAlready completed:",
    len(completed),
    "/",
    total_batches
)

if completed:

    print(
        "Completed batches:",
        [x + 1 for x in completed]
    )


# --------------------------------------------------
# PROCESS BATCHES
# --------------------------------------------------

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    # IMPORTANT:
    # Skip batches already saved
    if os.path.exists(batch_file):

        print(
            f"\nBatch {batch_idx + 1}/{total_batches} "
            f"already saved → SKIPPING"
        )

        continue


    start = batch_idx * BATCH_SIZE

    end = min(
        start + BATCH_SIZE,
        len(train_df)
    )

    batch_df = train_df.iloc[start:end]


    print("\n" + "=" * 60)

    print(
        f"Processing batch "
        f"{batch_idx + 1}/{total_batches}"
    )

    print(
        f"Files {start} to {end - 1}"
    )

    print("=" * 60)


    batch_embeddings = []

    batch_indices = []


    # --------------------------------------------------
    # PROCESS EACH AUDIO FILE
    # --------------------------------------------------

    for idx, row in batch_df.iterrows():

        source = row["source"]

        file_path = row["file_path"]


        # Special handling for pump paths
        if source == "0_dB_pump":

            marker = "0_dB_pump\\"

            if marker in file_path:

                file_path = file_path.split(
                    marker,
                    1
                )[1]

            file_path = file_path.replace(
                "\\",
                "/"
            )


        z = get_zip(source)


        # Read ONLY the required WAV
        with z.open(file_path) as audio_file:

            audio_bytes = audio_file.read()


        audio, sr = sf.read(
            io.BytesIO(audio_bytes),
            dtype="float32"
        )


        # Convert multi-channel audio to mono
        if audio.ndim > 1:

            audio = audio.mean(axis=1)


        waveform = torch.tensor(
            audio,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)


        # BEATs feature extraction
        with torch.no_grad():

            features, padding_mask = (
                beats.extract_features(
                    waveform,
                    padding_mask=None
                )
            )


        # Mean pooling
        embedding = features.mean(dim=1)

        embedding = (
            embedding
            .squeeze(0)
            .cpu()
            .numpy()
        )


        batch_embeddings.append(
            embedding
        )

        batch_indices.append(idx)


        if len(batch_embeddings) % 10 == 0:

            print(
                f"  "
                f"{len(batch_embeddings)}/"
                f"{len(batch_df)} processed"
            )


    # --------------------------------------------------
    # SAVE BATCH
    # --------------------------------------------------

    batch_embeddings = np.stack(
        batch_embeddings
    )

    batch_indices = np.array(
        batch_indices
    )


    np.savez_compressed(
        batch_file,
        embeddings=batch_embeddings,
        indices=batch_indices
    )


    # --------------------------------------------------
    # UPDATE PROGRESS
    # --------------------------------------------------

    completed_now = []

    for b in range(total_batches):

        f = os.path.join(
            output_dir,
            f"batch_{b:04d}.npz"
        )

        if os.path.exists(f):

            completed_now.append(b)


    with open(progress_file, "w") as f:

        json.dump(
            {
                "total_batches": total_batches,
                "completed_batches": completed_now,
                "total_files": len(train_df)
            },
            f,
            indent=2
        )


    print(
        f"\n✅ Batch "
        f"{batch_idx + 1} SAVED!"
    )

    print(
        "Embedding shape:",
        batch_embeddings.shape
    )

    print(
        "Progress:",
        len(completed_now),
        "/",
        total_batches
    )


# --------------------------------------------------
# FINISHED
# --------------------------------------------------

print("\n" + "=" * 60)

print("🎉 ALL BATCHES COMPLETE!")

print("=" * 60)

In [ ]:
import os

print("=== CHECKING CURRENT SESSION ===")

print("\nBEATs checkpoint candidates:")
for root, dirs, files in os.walk("/kaggle"):
    for f in files:
        if "BEATs_iter3" in f or f.endswith(".pt"):
            print(os.path.join(root, f))

print("\nExisting embedding files:")
embed_dir = "/kaggle/working/beats_train_embeddings"

if os.path.exists(embed_dir):
    files = sorted(os.listdir(embed_dir))
    print("Count:", len(files))
    print(files[:20])
else:
    print("Embedding directory does not exist yet.")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

In [ ]:
import os
import sys

# Clone BEATs source code if it is not already present
if not os.path.exists("/kaggle/working/unilm/beats"):
    !git clone https://github.com/microsoft/unilm.git /kaggle/working/unilm

# Put BEATs source first
sys.path.insert(0, "/kaggle/working/unilm/beats")

# Import backbone first — required by BEATs
import backbone
from BEATs import BEATs, BEATsConfig

print("✅ BEATs source code loaded!")

In [ ]:
import os

print("BEATs folders/files:")

for root, dirs, files in os.walk("/kaggle/working/unilm"):
    if "BEATs.py" in files or "backbone.py" in files:
        print("\nFolder:", root)
        for f in files:
            if f in ["BEATs.py", "backbone.py", "__init__.py"]:
                print("  ", f)

In [ ]:
import sys

# Use the actual BEATs source directory
beats_source = "/kaggle/working/unilm/beats"

if beats_source not in sys.path:
    sys.path.insert(0, beats_source)

# Import backbone first
import backbone
from BEATs import BEATs, BEATsConfig

print("✅ BEATs source imported successfully!")

In [ ]:
import torch

checkpoint_path = "/kaggle/input/datasets/nikhilns718/beats-iter3-plus-as2m/BEATs_iter3_plus_AS2M.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

cfg = BEATsConfig(checkpoint["cfg"])

beats = BEATs(cfg)
beats.load_state_dict(checkpoint["model"])
beats.to(device)
beats.eval()

print("✅ BEATs checkpoint loaded successfully!")

In [ ]:
import pandas as pd
import io
import fsspec
import zipfile
import soundfile as sf
import torch

# Load training metadata
train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"
train_df = pd.read_csv(train_csv)

row = train_df.iloc[0]

print("Machine:", row["machine"])
print("Label:", row["label"])
print("Source:", row["source"])
print("File:", row["file_path"])

In [ ]:
import fsspec
import zipfile
import io
import soundfile as sf
import torch
import numpy as np

source = row["source"]
file_path = row["file_path"]

# Open the remote ZIP without downloading the whole ZIP
zip_name = source
url = f"https://zenodo.org/records/3384388/files/{zip_name}?download=1"

print("Opening:", zip_name)

remote_file = fsspec.open(
    url,
    mode="rb",
    block_size=5 * 1024 * 1024
).open()

z = zipfile.ZipFile(remote_file)

print("Reading:", file_path)

with z.open(file_path) as audio_file:
    audio_bytes = audio_file.read()

audio, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

print("Sample rate:", sr)
print("Original shape:", audio.shape)

# Convert multi-channel audio to mono
if audio.ndim > 1:
    audio = audio.mean(axis=1)

waveform = torch.tensor(
    audio,
    dtype=torch.float32,
    device=device
).unsqueeze(0)

print("Waveform shape:", waveform.shape)

# BEATs feature extraction
with torch.no_grad():
    features, padding_mask = beats.extract_features(
        waveform,
        padding_mask=None
    )

print("BEATs feature shape:", features.shape)

# Mean pooling
embedding = features.mean(dim=1)

print("Embedding shape:", embedding.shape)
print("Embedding dimension:", embedding.shape[-1])

print("✅ Single-file BEATs test successful!")

In [ ]:
import os
import io
import time
import json
import zipfile
import fsspec
import torch
import soundfile as sf
import numpy as np
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

BATCH_SIZE = 100

train_csv = "/kaggle/input/datasets/nikhilns718/mimii-fault-detection-splits/train.csv"

output_dir = "/kaggle/working/beats_train_embeddings"
os.makedirs(output_dir, exist_ok=True)

progress_file = os.path.join(output_dir, "progress.json")

# ============================================================
# LOAD TRAINING DATA
# ============================================================

train_df = pd.read_csv(train_csv)

total_files = len(train_df)
total_batches = int(np.ceil(total_files / BATCH_SIZE))

print("Total training files:", total_files)
print("Batch size:", BATCH_SIZE)
print("Total batches:", total_batches)

# ============================================================
# REMOTE ZIP CACHE
# ============================================================

zip_files = {}

def get_zip(source):

    if source in zip_files:
        return zip_files[source]

    zip_name = source if source.endswith(".zip") else source + ".zip"

    url = (
        f"https://zenodo.org/records/3384388/"
        f"files/{zip_name}?download=1"
    )

    print("\nOpening:", zip_name)

    remote_file = fsspec.open(
        url,
        mode="rb",
        block_size=5 * 1024 * 1024
    ).open()

    z = zipfile.ZipFile(remote_file)

    zip_files[source] = z

    return z


# ============================================================
# RETRY FUNCTION FOR ZENODO 429 / NETWORK ERRORS
# ============================================================

def get_audio_bytes(source, file_path, max_retries=8):

    for attempt in range(max_retries):

        try:

            z = get_zip(source)

            with z.open(file_path) as audio_file:
                return audio_file.read()

        except Exception as e:

            error_text = str(e)

            print(
                f"\n⚠️ Error reading {file_path}"
                f"\nAttempt {attempt + 1}/{max_retries}"
                f"\n{error_text[:300]}"
            )

            # Close cached ZIP if something went wrong
            if source in zip_files:

                try:
                    zip_files[source].close()
                except:
                    pass

                del zip_files[source]

            # Exponential backoff
            wait_time = min(60, 10 * (attempt + 1))

            print(f"Waiting {wait_time} seconds before retry...")

            time.sleep(wait_time)

    raise RuntimeError(
        f"Failed after {max_retries} attempts:\n"
        f"Source: {source}\n"
        f"File: {file_path}"
    )


# ============================================================
# FIND ALREADY COMPLETED BATCHES
# ============================================================

completed_batches = []

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    if os.path.exists(batch_file):

        completed_batches.append(batch_idx)


print(
    "\nAlready completed:",
    len(completed_batches),
    "/",
    total_batches
)

if completed_batches:

    print(
        "Completed batches:",
        [x + 1 for x in completed_batches]
    )


# ============================================================
# MAIN EXTRACTION LOOP
# ============================================================

for batch_idx in range(total_batches):

    batch_file = os.path.join(
        output_dir,
        f"batch_{batch_idx:04d}.npz"
    )

    # --------------------------------------------------------
    # SKIP COMPLETED BATCH
    # --------------------------------------------------------

    if os.path.exists(batch_file):

        print(
            f"\nBatch {batch_idx + 1}/{total_batches}"
            " already saved → SKIPPING"
        )

        continue


    # --------------------------------------------------------
    # BATCH RANGE
    # --------------------------------------------------------

    start = batch_idx * BATCH_SIZE
    end = min(
        start + BATCH_SIZE,
        total_files
    )

    batch_df = train_df.iloc[start:end]

    print("\n" + "=" * 70)

    print(
        f"PROCESSING BATCH "
        f"{batch_idx + 1}/{total_batches}"
    )

    print(
        f"Files {start} to {end - 1}"
    )

    print("=" * 70)


    batch_embeddings = []
    batch_indices = []


    # --------------------------------------------------------
    # PROCESS EACH AUDIO FILE
    # --------------------------------------------------------

    for local_count, (idx, row) in enumerate(
        batch_df.iterrows(),
        start=1
    ):

        source = row["source"]
        file_path = row["file_path"]


        # ----------------------------------------------------
        # SPECIAL PUMP PATH FIX
        # ----------------------------------------------------

        if source == "0_dB_pump":

            marker = "0_dB_pump\\"

            if marker in file_path:

                file_path = file_path.split(
                    marker,
                    1
                )[1]

            file_path = file_path.replace(
                "\\",
                "/"
            )


        # ----------------------------------------------------
        # READ AUDIO WITH RETRY
        # ----------------------------------------------------

        audio_bytes = get_audio_bytes(
            source,
            file_path
        )


        # ----------------------------------------------------
        # DECODE AUDIO
        # ----------------------------------------------------

        audio, sr = sf.read(
            io.BytesIO(audio_bytes),
            dtype="float32"
        )


        # ----------------------------------------------------
        # CONVERT MULTI-CHANNEL → MONO
        # ----------------------------------------------------

        if audio.ndim > 1:

            audio = audio.mean(axis=1)


        # ----------------------------------------------------
        # PYTORCH WAVEFORM
        # ----------------------------------------------------

        waveform = torch.tensor(
            audio,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)


        # ----------------------------------------------------
        # BEATs FEATURE EXTRACTION
        # ----------------------------------------------------

        with torch.no_grad():

            features, padding_mask = (
                beats.extract_features(
                    waveform,
                    padding_mask=None
                )
            )


        # ----------------------------------------------------
        # TEMPORAL MEAN POOLING
        # ----------------------------------------------------

        embedding = features.mean(
            dim=1
        )

        embedding = (
            embedding
            .squeeze(0)
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        batch_embeddings.append(
            embedding
        )

        batch_indices.append(
            idx
        )


        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if local_count % 10 == 0:

            print(
                f"  {local_count}/{len(batch_df)} "
                f"processed"
            )


    # ========================================================
    # SAVE BATCH
    # ========================================================

    batch_embeddings = np.stack(
        batch_embeddings
    )

    batch_indices = np.array(
        batch_indices
    )


    np.savez_compressed(
        batch_file,
        embeddings=batch_embeddings,
        indices=batch_indices
    )


    # ========================================================
    # UPDATE PROGRESS
    # ========================================================

    completed_now = []

    for b in range(total_batches):

        f = os.path.join(
            output_dir,
            f"batch_{b:04d}.npz"
        )

        if os.path.exists(f):

            completed_now.append(b)


    with open(
        progress_file,
        "w"
    ) as f:

        json.dump(
            {
                "total_batches": total_batches,
                "completed_batches":
                    completed_now,
                "total_files": total_files
            },
            f,
            indent=2
        )


    print(
        f"\n✅ BATCH {batch_idx + 1} SAVED!"
    )

    print(
        "Embedding shape:",
        batch_embeddings.shape
    )

    print(
        "Progress:",
        len(completed_now),
        "/",
        total_batches
    )


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("🎉 ALL BEATs EMBEDDINGS COMPLETE!")
print("=" * 70)

print(
    "Total files:",
    total_files
)

print(
    "Output directory:",
    output_dir
)

In [ ]:
import os
import shutil

output_dir = "/kaggle/working/beats_train_embeddings"
checkpoint_zip = "/kaggle/working/BEATs_ALL_4419_embeddings"

print("Checking embedding files...")

files = sorted(os.listdir(output_dir))

batch_files = [
    f for f in files
    if f.startswith("batch_") and f.endswith(".npz")
]

print("Batch files:", len(batch_files))
print("Expected:", 45)

# Create ZIP
print("\nCreating checkpoint ZIP...")

shutil.make_archive(
    checkpoint_zip,
    "zip",
    output_dir
)

zip_path = checkpoint_zip + ".zip"

size_mb = os.path.getsize(zip_path) / (1024 ** 2)

print("\n✅ CHECKPOINT CREATED!")
print("File:", zip_path)
print(f"Size: {size_mb:.2f} MB")